# Lab 3  Build a ReAct Agent (tool use) + the SOTA way

**Course:** Agentic AI  **Prereqs:** Labs 1–2

In Lab 1 the policy picked a grid move in Lab 2 an LLM picked the move. Now the LLM uses tools to solve open-ended tasks with the ReAct loop: Thought → Action → Observation, repeat, until a Final Answer. You build the loop by hand  no LangChain. Then you see the production way: Gemini native function calling, where the whole loop collapses into `tools=[...]`.


**The six TODOs:** (1) `parse()`, (2) `run()` the loop, (3) `run_tool()` the guardrail, (4) the `word_count` tool, (5) native auto function calling, (6) native manual dispatch.

## 0 · Setup

In [1]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install python-dotenv

In [3]:
import os
from dotenv import load_dotenv

# Загружаем переменные из .env файла
load_dotenv()

# Теперь ключ доступен через os.getenv
api_key = os.getenv("GEMINI_API_KEY")
print("API key set:", bool(api_key))

API key set: True


## 1 · Tools

Each tool is a plain function `str -> str`. `TOOLS` maps a name to `(function, description)` the description goes into the prompt so the model knows how to call it.

**Safety:** a tool's input comes from the *model*, so `calculator` parses with `ast` and only allows a whitelist of math — never `eval`. **TODO:** finish `word_count`.

In [4]:
import ast, operator, datetime

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod, ast.FloorDiv: operator.floordiv,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _eval(node):
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported or unsafe expression")

def calculator(expr: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. 2*(3+4)."""
    try:
        return str(_eval(ast.parse(expr, mode="eval")))
    except Exception as e:
        return f"calculator error: {e}"

def today(_: str = "") -> str:
    """Return today's date in ISO format (ignores its input)."""
    return datetime.date.today().isoformat()

def wikipedia(query: str) -> str:
    """Return a short factual summary from Wikipedia's REST API (needs internet)."""
    import requests, urllib.parse
    title = urllib.parse.quote(query.strip().replace(" ", "_"))
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": "agentic-ai-course/1.0"})
        if r.status_code == 404:
            return f"No Wikipedia page found for {query!r}."
        r.raise_for_status()
        return (r.json().get("extract") or "(no summary)")[:600]
    except Exception as e:
        return f"wikipedia error: {e}"

def word_count(text: str) -> str:
    """Return the number of whitespace-separated words in text (as a string)."""
    # TODO: split `text` on whitespace, count the words, return the count AS A STRING.
    return str(len(text.split()))

TOOLS = {
    "calculator": (calculator, "evaluate arithmetic, e.g. calculator[2*(3+4)]"),
    "today": (today, "return today's date in ISO format, e.g. today[]"),
    "wikipedia": (wikipedia, "look up a short factual summary, e.g. wikipedia[Alan Turing]"),
    "word_count": (word_count, "count words in text, e.g. word_count[the quick brown fox]"),
}

## 2 · Talking to Gemini

In [5]:
import time
_client = None
MODEL = "gemini-2.5-flash"

def chat(messages, stop=None, temperature: float = 0.0, max_tokens: int = 1024) -> str:
    global _client
    from google import genai
    from google.genai import types
    if _client is None:
        _client = genai.Client()
    system = "\n".join(m["content"] for m in messages if m["role"] == "system") or None
    contents = [{"role": ("model" if m["role"] == "assistant" else "user"), "parts": [{"text": m["content"]}]}
                for m in messages if m["role"] != "system"]
    cfg = types.GenerateContentConfig(system_instruction=system, temperature=temperature,
                                      max_output_tokens=max_tokens, stop_sequences=stop or None)
    delay = 2.0
    for attempt in range(4):
        try:
            return _client.models.generate_content(model=MODEL, contents=contents, config=cfg).text or ""
        except Exception as e:
            if any(t in str(e) for t in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and attempt < 3:
                time.sleep(delay); delay *= 2; continue
            raise

## 3 · The ReAct agent

The model emits `Thought:` / `Action: tool[input]` runs the tool and feeds back `Observation:` repeat until `Final Answer:`. `stop=["Observation:"]` makes the model halt before hallucinating its own observation. `max_steps` can be used as well.

Fill **TODO** (`parse`), **TODO** (`run_tool`), **TODO** (`run`). The regexes, prompt, and dataclasses are provided.

In [6]:
import re
from dataclasses import dataclass, field
from typing import Callable, Optional

ACTION_RE = re.compile(r"Action:\s*([A-Za-z_]\w*)\s*\[(.*?)\]", re.DOTALL)
FINAL_RE = re.compile(r"Final Answer:\s*(.*)", re.DOTALL)

SYSTEM_TEMPLATE = """You are a reasoning agent that solves tasks step by step using tools.

Always reply in EXACTLY this format, one step at a time:

Thought: <your reasoning>
Action: <tool_name>[<input>]

You will then be shown:
Observation: <result of the action>

Repeat Thought/Action as needed. When ready to answer, reply:

Thought: <final reasoning>
Final Answer: <the answer>

Available tools:
{tool_list}

Rules:
- Emit ONE Thought and then ONE Action (or a Final Answer). Never write the Observation yourself.
- Only use a tool from the list above, and follow its example format exactly.
"""

@dataclass
class Step:
    raw: str
    action: Optional[str] = None
    action_input: Optional[str] = None
    observation: Optional[str] = None

@dataclass
class Result:
    answer: Optional[str]
    steps: list = field(default_factory=list)
    stopped_reason: str = "final_answer"

def _system_prompt(tools) -> str:
    tool_list = "\n".join(f"- {name}[input]: {desc}" for name, (_, desc) in tools.items())
    return SYSTEM_TEMPLATE.format(tool_list=tool_list)

def parse(text: str):
    fm = FINAL_RE.search(text)
    am = ACTION_RE.search(text)
    if fm and am:
        if fm.start() < am.start():
            return ("final", fm.group(1).strip())
        else:
            return ("action", am.group(1), am.group(2))
    if fm:
        return ("final", fm.group(1).strip())
    if am:
        return ("action", am.group(1), am.group(2))
    return ("error", None)

def run_tool(action: str, action_input: str, tools=TOOLS) -> str:
    if action not in tools:
        return f"Unknown tool"
    func, desc = TOOLS[action]
    return func(action_input)

def run(task: str, llm: Callable = None, tools=TOOLS, max_steps: int = 8, verbose: bool = True) -> Result:
    """TODO: the ReAct loop.
    Keep `llm` injectable  the self-check passes a fake, so do NOT hard-code chat."""
    if llm is None: llm = chat
    messages = [
        {"role": "system", "content": _system_prompt(tools) },
        {"role": "user", "content": task}
    ]
    steps = []
    for _ in range(max_steps):
        response = llm(messages)
        if verbose:
            print(response)
        kind = parse(response)
        if kind[0] == "final":
            return Result(answer=kind[1], steps=steps, stopped_reason="final_answer")
        if kind[0] == "error":
            return Result(answer=None, steps=steps,stopped_reason="parse_error")
        _, action, action_input = kind
        observation = run_tool(action, action_input, tools)
        step = Step(
            raw=response,
            action=action,
            action_input=action_input,
            observation=observation
        )
        steps.append(step)
        messages.append(
            {"role": "assistant", "content": response }
        )
        messages.append(
            {"role": "user","content": f"Observation: {observation}"}
        )
    return Result(
        answer=None,
        steps=steps,
        stopped_reason="max_steps"
    )


## 5 · Run it live on Gemini

In [7]:
res = run("What is the capital of USA?")
print("\n=== FINAL ANSWER ===")
print(res.answer, f"  (stopped: {res.stopped_reason}, steps: {len(res.steps)})")

Action: wikipedia[capital of USA]
Thought: The previous search for "capital of USA" did not yield a result. I should try a more direct query, such as "United States capital".
Action: wikipedia[United States capital]
Thought: The Wikipedia search successfully identified Washington, D.C. as the capital city of the United States.
Final Answer: Washington, D.C.

=== FINAL ANSWER ===
Washington, D.C.   (stopped: final_answer, steps: 2)


## 6 · Native function calling

You hand-wrote the loop but production agents hand the model the tool **functions** and let the SDK exchange **structured** tool calls. With Gemini **Automatic Function Calling (AFC)**, the whole loop becomes `tools=[...]`.

- **TODO:** list the tool functions for AFC.
- **TODO:** `dispatch_calls` run the model's structured calls locally (the native analogue of `run_tool`).

In [8]:
_FUNCS = {"calculator": calculator, "today": today, "wikipedia": wikipedia, "word_count": word_count}

def run_native_auto(task: str, model: str = None) -> str:
    """Automatic Function Calling: the SDK runs the whole loop for you."""
    if model is None: model = MODEL
    from google import genai
    from google.genai import types
    client = genai.Client()
    resp = client.models.generate_content(
        model=model, contents=task,
        config=types.GenerateContentConfig(
            temperature=0.0,
            # TODO: list the FUNCTION OBJECTS (but do not strings) so the model can call them.
            tools=[calculator, today, wikipedia, word_count],   # <-- fill in: calculator, today, wikipedia, word_count
        ),
    )
    return resp.text or ""

def dispatch_calls(function_calls):
    """TODO 6 for each call (it has .name and .args, a dict): look up _FUNCS[name] if missing append
    (name, "unknown tool ...")  else append (name, func(**dict(fc.args or {}))). Return the list."""
    out = []
    for fc in function_calls:
        if fc.name not in _FUNCS:
            out.append((fc.name, f"unknown tool {fc.name}"))
        else:
            out.append((fc.name, _FUNCS[fc.name](**dict(fc.args or {}))))
    return out

def run_native_manual(task: str, model: str = None):
    """Manual mode: disable AFC so you SEE the structured calls the model makes, then run them."""
    if model is None: model = MODEL
    from google import genai
    from google.genai import types
    client = genai.Client()
    resp = client.models.generate_content(
        model=model, contents=task,
        config=types.GenerateContentConfig(
            temperature=0.0,
            tools=[calculator, today, wikipedia, word_count],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )
    calls = resp.function_calls or []
    if not calls:
        return resp.text or "(model answered directly)"
    for name, output in dispatch_calls(calls):
        print(f"  model requested {name}(...) -> {output}")

### Self-check for `dispatch_calls`

In [9]:
class _FC:
    def __init__(self, name, args): self.name, self.args = name, args

assert dispatch_calls([_FC("calculator", {"expr": "2*21"}), _FC("word_count", {"text": "a b c d"})]) \
       == [("calculator", "42"), ("word_count", "4")]
assert "unknown" in dispatch_calls([_FC("nope", {})])[0][1]
print("DISPATCH OK")

DISPATCH OK


### Run native function calling live

In [10]:
print("AUTOMATIC (SDK runs the loop):")
print(run_native_auto("What is 24*7, and what day is it today?"))

print("\nMANUAL (you see and run the structured calls):")
run_native_manual("How many words are in 'the quick brown fox', and what is 19*3?")

AUTOMATIC (SDK runs the loop):
24 multiplied by 7 is 168. Today is 2026-06-27.

MANUAL (you see and run the structured calls):
  model requested word_count(...) -> 4
  model requested calculator(...) -> 57


## 7 · Report (deliverable)
Answer in a text cell below
1. **The loop.** Trace one Thought→Action→Observation→Final Answer cycle through your `run()`. What does `stop=["Observation:"]` prevent?
2. **Safety** What does the agent do (instead of crashing) on an unknown tool and on an unparseable reply?
3. **Tool safety.** Why does `calculator` use `ast`, not `eval`? Give one dangerous input.
4. **Hand-rolled vs native.** Two things native function calling gives you over text-parsed ReAct, and one thing the hand-rolled version makes visible that the automatic one hides. When use each?

**References:** ReAct (Yao et al., ICLR 2023, arXiv:2210.03629); "Building Effective Agents" (2024); Gemini function calling (ai.google.dev/gemini-api/docs/function-calling).